## Database Connection (Config)

In [1]:
!pip install -q pymongo dnspython plotly pandas python-dotenv

import os
import json
from pymongo import MongoClient
from bson import json_util
from dotenv import load_dotenv

load_dotenv()

URI = os.getenv('MONGO_URI')
DB_NAME = os.getenv('DB_NAME', 'tracera')
COLLECTION_NAME = os.getenv('COLLECTION_NAME', 'raw_linkedin')

def connect_to_mongodb_and_get_data(uri, db_name, collection_name, query={}):
    if not uri:
        raise ValueError("Kredensial MONGO_URI tidak ditemukan! Pastikan file .env sudah ada.")
    try:
        client = MongoClient(uri)
        db = client[db_name]
        collection = db[collection_name]
        documents = list(collection.find(query))
        json_data = json.loads(json_util.dumps(documents))
        print(f"Berhasil terhubung! {len(json_data)} dokumen dimuat dari koleksi '{collection_name}'.")
        return json_data
    except Exception as e:
        print(f"Error koneksi: {e}")
        return []

raw_documents = connect_to_mongodb_and_get_data(URI, DB_NAME, COLLECTION_NAME)



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Berhasil terhubung! 397 dokumen dimuat dari koleksi 'raw_linkedin'.


## Pre-processing Data

In [2]:
from datetime import datetime, timezone
import pandas as pd

# ==============================================================================
# 1. KAMUS TRANSLASI & NORMALISASI SKILLS (Bilingual & Canonical Normalization)
# ==============================================================================
SKILL_TRANSLATIONS = {
    # a. Hard Skills - Desain, Media & Kreatif Teknis
    'desain': 'Design',
    'design': 'Design',
    'desain grafis': 'Graphic Design',
    'design grafis': 'Graphic Design',
    'graphic design': 'Graphic Design',
    'pemikiran desain': 'Design Thinking',
    'creative direction': 'Creative Direction',
    'creative strategy': 'Creative Strategy',
    'penulisan': 'Writing',
    'writing': 'Writing',
    'creative writing': 'Creative Writing',
    'penulisan kreatif': 'Creative Writing',
    'report writing': 'Report Writing',
    'penulisan laporan': 'Report Writing',
    'technical writing': 'Technical Writing',
    'penulisan teknis': 'Technical Writing',
    'scientific writing': 'Scientific Writing',
    'penulisan ilmiah': 'Scientific Writing',
    'proposal writing': 'Proposal Writing',
    'penulisan proposal': 'Proposal Writing',

    # b. Hard Skills - Data & Analisis
    'analisis data': 'Data Analysis',
    'analisa data': 'Data Analysis',
    'analisis': 'Data Analysis',
    'data analysis': 'Data Analysis',
    'analisis bisnis': 'Business Analysis',
    'business analysis': 'Business Analysis',
    'analisis keuangan': 'Financial Analysis',
    'financial analysis': 'Financial Analysis',
    'analisis sosial': 'Social Analysis',
    'pengolahan data': 'Data Processing',
    'entri data': 'Data Entry',
    'data entry': 'Data Entry',
    'mangalisis suatu proyek': 'Project Analysis',
    'riset': 'Research',
    'penelitian': 'Research',
    'research': 'Research',
    'research skills': 'Research',
    'analytical skills': 'Analytical Skills',
    'keahlian analitis': 'Analytical Skills',
    'analytic problem solving': 'Analytical Skills',

    # c. Hard Skills - Manajemen Proyek, Operasional & IT / Tools
    'manajemen proyek': 'Project Management',
    'project management': 'Project Management',
    'dokumen proyek': 'Project Documentation',
    'manajemen operasional': 'Operations Management',
    'operations management': 'Operations Management',
    'pengembangan web': 'Web Development',
    'manajemen file': 'File Management',
    'sistem informasi geografis (gis)': 'GIS',
    'geographic information systems (gis)': 'GIS',
    'arcgis': 'GIS',
    'sistem manajemen keselamatan dan kesehatan kerja': 'Occupational Health and Safety (OHS)',
    'kesehatan, keselamatan, dan lingkungan (hse)': 'Health, Safety, and Environment (HSE)',
    'peningkatan proses bisnis': 'Business Process Improvement',
    'monitoring dan evaluasi': 'Monitoring & Evaluation',
    'perencanaan strategis': 'Strategic Planning',
    'strategic planning': 'Strategic Planning',
    'manajemen perubahan': 'Change Management',
    'change management': 'Change Management',
    'organizational development': 'Organizational Development',
    'pengembangan organisasi': 'Organizational Development',
    'collaboration solutions': 'Collaboration Tools',
    'python (programming language)': 'Python',
    'go (programming language)': 'Go',
    'golang': 'Go',
    'c (programming language)': 'C',
    'r (programming language)': 'R',
    'java (programming language)': 'Java',
    'powerpoint': 'Microsoft PowerPoint',
    'excel': 'Microsoft Excel',
    'word': 'Microsoft Word',

    # d. Hard Skills - Keuangan, Akuntansi & Hukum Bisnis
    'keuangan': 'Finance',
    'finance': 'Finance',
    'jasa keuangan': 'Financial Services',
    'manajemen keuangan': 'Financial Management',
    'financial management': 'Financial Management',
    'perencanaan keuangan': 'Financial Planning',
    'financial planning': 'Financial Planning',
    'akuntansi': 'Accounting',
    'accounting': 'Accounting',
    'akuntansi keuangan': 'Financial Accounting',
    'financial accounting': 'Financial Accounting',
    'pembukuan': 'Bookkeeping',
    'bookkeeping': 'Bookkeeping',
    'perpajakan': 'Taxation',
    'tax': 'Taxation',
    'taxations': 'Taxation',
    'perencanaan anggaran': 'Budgeting',
    'budgeting': 'Budgeting',
    'budgets': 'Budgeting',
    'budget management': 'Budgeting',
    'auditing': 'Auditing',
    'internal audits': 'Auditing',
    'financial audits': 'Auditing',
    'financial reporting': 'Financial Reporting',
    'financial statements': 'Financial Reporting',
    'mengolah data biaya': 'Cost Accounting',
    'cost accounting': 'Cost Accounting',
    'konsultan manajemen': 'Management Consulting',
    'konsultan lingkungan': 'Environmental Consulting',
    'hukum': 'Legal & Law',
    'negosiasi kontrak': 'Contract Negotiation',
    'contract negotiation': 'Contract Negotiation',

    # e. Hard Skills - Pemasaran, Komunikasi Korporat, Penjualan & SDM
    'pemasaran': 'Marketing',
    'marketing': 'Marketing',
    'pemasaran digital': 'Digital Marketing',
    'pemasaran media sosial': 'Social Media Marketing',
    'social media marketing': 'Social Media Marketing',
    'pemasaran produk': 'Product Marketing',
    'strategi pemasaran': 'Marketing Strategy',
    'marketing strategy': 'Marketing Strategy',
    'strategi konten': 'Content Strategy',
    'content strategy': 'Content Strategy',
    'pengoptimalan media sosial (social media optimization/smo)': 'Social Media Optimization (SMO)',
    'komunikasi merek': 'Brand Communication',
    'brand communication': 'Brand Communication',
    'social media communications': 'Social Media Marketing',
    'hubungan masyarakat': 'Public Relations',
    'public relations': 'Public Relations',
    'penjualan': 'Sales',
    'sales': 'Sales',
    'perencanaan acara': 'Event Planning',
    'event planning': 'Event Planning',
    'event management': 'Event Planning',
    'sumber daya manusia (sdm)': 'Human Resources (HR)',
    'human resources (hr)': 'Human Resources (HR)',
    'human resources': 'Human Resources (HR)',
    'perencanaan sumber daya manusia strategis': 'Strategic Human Resource Planning',
    'administrasi': 'Administration',
    'administration': 'Administration',
    'office administration': 'Administration',
    'administrasi perkantoran': 'Administration',
    'bekerja di shift': 'Shift Work',
    'opname kerja': 'Stock Opname',

    # f. Hard Skills - Bahasa & Pengajaran
    'english': 'English',
    'bahasa inggris': 'English',
    'bahasa indonesia': 'Indonesian',
    'teaching': 'Teaching',
    'pengajaran': 'Teaching',

    # g. Soft Skills - Komunikasi & Interpersonal
    'komunikasi': 'Communication',
    'communication': 'Communication',
    'communications': 'Communication',
    'kemampuan komunikasi': 'Communication',
    'interpersonal communication': 'Communication',
    'written communication': 'Communication',
    'strategic communications': 'Communication',
    'intercultural communication': 'Communication',
    'cross-cultural communication skills': 'Communication',
    'komunikasi interpersonal': 'Communication',
    'komunikasi tertulis': 'Communication',
    'active listening': 'Active Listening',
    'mendengarkan aktif': 'Active Listening',
    'persuasi': 'Persuasion',
    'persuasion': 'Persuasion',
    'layanan pelanggan': 'Customer Service',
    'customer service': 'Customer Service',
    'pelayanan': 'Customer Service',
    'keahlian interpersonal': 'Interpersonal Skills',
    'interpersonal skills': 'Interpersonal Skills',
    'interpersonal': 'Interpersonal Skills',
    'keahlian presentasi': 'Presentation Skills',
    'presentation skills': 'Presentation Skills',
    'presentation': 'Presentation Skills',
    'presentations': 'Presentation Skills',
    'presentasi': 'Presentation Skills',
    'public speaking': 'Public Speaking',
    'berbicara di depan umum': 'Public Speaking',
    'relationship building': 'Relationship Building',
    'membangun relasi': 'Relationship Building',

    # h. Soft Skills - Tim & Kerjasama
    'kerja sama tim': 'Teamwork',
    'kerja tim': 'Teamwork',
    'kerjasama tim': 'Teamwork',
    'kemampuan kerja tim': 'Teamwork',
    'teamwork': 'Teamwork',
    'team work': 'Teamwork',
    'kolaborasi tim': 'Collaboration',
    'collaboration': 'Collaboration',
    'team collaboration': 'Collaboration',
    'cross-team collaboration': 'Collaboration',
    'cross-functional collaborations': 'Collaboration',
    'resolusi konflik': 'Conflict Resolution',
    'conflict resolution': 'Conflict Resolution',
    'conflict management': 'Conflict Resolution',
    'manajemen konflik': 'Conflict Resolution',

    # i. Soft Skills - Kepemimpinan & Manajemen Orang
    'kepemimpinan': 'Leadership',
    'leadership': 'Leadership',
    'team leadership': 'Leadership',
    'kepemimpinan tim': 'Leadership',
    'project leadership': 'Leadership',
    'organizational leadership': 'Leadership',
    'strategic leadership': 'Leadership',
    'cross-functional team leadership': 'Leadership',
    'leadership development': 'Leadership Development',
    'pengembangan kepemimpinan': 'Leadership Development',
    'manajemen tim': 'Team Management',
    'team management': 'Team Management',
    'team coordination': 'Team Management',
    'team building': 'Team Management',
    'team motivation': 'Team Management',
    'people management': 'People Management',
    'manajemen orang': 'People Management',
    'people development': 'People Development',
    'team development': 'People Development',
    'pengembangan tim': 'People Development',
    'coaching': 'Coaching & Mentoring',
    'mentoring': 'Coaching & Mentoring',
    'coaching & mentoring': 'Coaching & Mentoring',
    'training & mentoring': 'Coaching & Mentoring',
    'pelatihan & mentoring': 'Coaching & Mentoring',
    'sales coaching': 'Coaching & Mentoring',
    'kemampuan berorganisasi': 'Organization Skills',
    'organization skills': 'Organization Skills',
    'organizational skills': 'Organization Skills',
    'manajemen': 'Management',
    'management': 'Management',

    # j. Soft Skills - Pemecahan Masalah & Berpikir Kritis
    'pemecahan masalah': 'Problem Solving',
    'memecahkan masalah': 'Problem Solving',
    'problem solving': 'Problem Solving',
    'creative problem solving': 'Problem Solving',
    'collaborative problem solving': 'Problem Solving',
    'team problem solving': 'Problem Solving',
    'berpikir kritis': 'Critical Thinking',
    'critical thinking': 'Critical Thinking',
    'critical thinker': 'Critical Thinking',
    'analytical thinking': 'Critical Thinking',
    'pengambilan keputusan': 'Decision Making',
    'decision making': 'Decision Making',
    'decision-making': 'Decision Making',
    'data-driven decision making': 'Decision Making',
    'strategic thinking': 'Strategic Thinking',
    'creativity': 'Creativity',
    'kreativitas': 'Creativity',
    'creative thinking': 'Creativity',
    'creativity skills': 'Creativity',
    'creativity and innovation': 'Creativity',

    # k. Soft Skills - Manajemen Diri & Sikap Kerja
    'manajemen waktu': 'Time Management',
    'time management': 'Time Management',
    'manajemen diri': 'Self Management',
    'self management': 'Self Management',
    'adaptability': 'Adaptability',
    'adaptable': 'Adaptability',
    'easily adaptable': 'Adaptability',
    'adaptation': 'Adaptability',
    'flexibility': 'Adaptability',
    'negosiasi': 'Negotiation',
    'negotiation': 'Negotiation',
    'negotiation skills': 'Negotiation',
    'emotional intelligence': 'Emotional Intelligence',
    'work ethic': 'Work Ethic',
    'etika kerja': 'Work Ethic',
    'working under pressure': 'Working Under Pressure',
    'bekerja dalam tekanan': 'Working Under Pressure',
    'hard working': 'Work Ethic',
    'bekerja keras': 'Work Ethic',
    'jujur dan disiplin': 'Honesty & Discipline',
    'honesty & discipline': 'Honesty & Discipline',
    'bintal dan jasmani': 'Mental & Physical Discipline',
    'mental & physical discipline': 'Mental & Physical Discipline',
    'organizational culture': 'Organizational Culture'
}

# ==============================================================================
# 2. TAKSONOMI RESMI SOFT SKILLS (Bentuk Kanonikal Hasil Normalisasi)
# ==============================================================================
# Disimpan dalam huruf kecil untuk matching case-insensitive terhadap s_norm
SOFT_SKILLS = {
    # Komunikasi & Interpersonal
    'communication',
    'public speaking',
    'presentation skills',
    'active listening',
    'persuasion',
    'customer service',
    'interpersonal skills',
    'relationship building',

    # Tim, Kerjasama, & Hubungan
    'teamwork',
    'collaboration',
    'conflict resolution',

    # Kepemimpinan & People Management
    'leadership',
    'leadership development',
    'team management',
    'people management',
    'people development',
    'coaching & mentoring',
    'organization skills',
    'management',
    'organizational culture',

    # Pemecahan Masalah & Berpikir Kritis
    'problem solving',
    'critical thinking',
    'decision making',
    'strategic thinking',
    'creativity',

    # Adaptabilitas, Manajemen Waktu, & Sikap Kerja
    'adaptability',
    'time management',
    'self management',
    'negotiation',
    'emotional intelligence',
    'work ethic',
    'working under pressure',
    'honesty & discipline',
    'mental & physical discipline'
}

# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================
def normalize_skill(name):
    """Membersihkan dan menstandardisasi nama keahlian."""
    if not name or not isinstance(name, str):
        return ''
    clean = name.strip()
    return SKILL_TRANSLATIONS.get(clean.lower(), clean)


def clean_job_title(title):
    """Membersihkan nama jabatan dari noise keterangan atau nama institusi."""
    if not title or not isinstance(title, str):
        return ''
    t = (
        title.split(' at ')[0]
        .split(' - ')[0]
        .split(' | ')[0]
        .split(' @ ')[0]
        .strip()
    )
    low = t.lower()
    noise_keywords = [
        'experience', 'endorsed', 'across', 'grade:', 'passed',
        'college', 'academy', 'universitas', 'gpa', 'alumni', "'--"
    ]
    if any(k in low for k in noise_keywords):
        return ''
    if len(t) < 3 or len(t) > 42:
        return ''
    return t

# ==============================================================================
# 4. FUNGSI PARSING DOKUMEN & PEMBERSIHAN ANOMALI
# ==============================================================================
def clean_doc(d):
    if not d or not isinstance(d, dict):
        return None

    raw_list = d.get('raw_data', [{}])
    p = raw_list[0] if isinstance(raw_list, list) and len(raw_list) > 0 else {}

    pid = p.get('publicIdentifier')
    if not pid or not isinstance(pid, str) or not pid.strip():
        return None

    # a. Validasi rentang tahun kerja & pembersihan jabatan
    experiences = []
    years = []
    for e in p.get('experience', []):
        yr = e.get('startDate', {}).get('year') if isinstance(e.get('startDate'), dict) else None
        if yr and str(yr).isdigit():
            yr_int = int(yr)
            if 1980 <= yr_int <= 2027:
                years.append(yr_int)
                clean_title = clean_job_title(e.get('position', ''))
                experiences.append({'year': yr_int, 'title': clean_title})

    # b. Industri (fallback ke 'Other')
    ind = p.get('industry')
    ind_name = ind.get('name') if isinstance(ind, dict) else (ind or 'Other')
    ind_clean = ind_name.strip() if isinstance(ind_name, str) and ind_name.strip() else 'Other'

    # c. Timestamp fetched_at
    fa = d.get('metadata', {}).get('fetched_at')
    try:
        fetched_dt = datetime.fromisoformat(fa) if fa else datetime.now(timezone.utc)
    except Exception:
        fetched_dt = datetime.now(timezone.utc)

    # d. Normalisasi Bahasa & Deduplikasi Skills per Profil
    skill_map = {}
    all_raw_s = [s['name'] for s in p.get('skills', []) if isinstance(s, dict) and s.get('name')]
    all_raw_s += (p.get('topSkills') or [])

    for s in all_raw_s:
        s_norm = normalize_skill(s)
        if len(s_norm) > 1:
            key = s_norm.lower()
            if key not in skill_map:
                skill_map[key] = s_norm
            elif s_norm[0].isupper() and not skill_map[key][0].isupper():
                skill_map[key] = s_norm

    # e. Program studi
    progs = []
    for e in p.get('education', []):
        fos = e.get('fieldOfStudy')
        if fos and isinstance(fos, str):
            fos_clean = fos.strip()
            if len(fos_clean) > 2 and fos_clean.lower() not in ['-', '.', '...', 'null', 'none']:
                if fos_clean not in progs:
                    progs.append(fos_clean)

    return {
        'id': pid.strip(),
        'industry': ind_clean,
        'fetched_at': fetched_dt,
        'skills': list(skill_map.values()),
        'programs': progs,
        'experiences': experiences,
        'latest_work_year': max(years, default=None)
    }

# ==============================================================================
# 5. BUAT DATAFRAME UTAMA & DEDUPLIKASI PROFIL
# ==============================================================================
raw_parsed = [clean_doc(d) for d in raw_documents if d]
raw_parsed = [doc for doc in raw_parsed if doc is not None]

df_raw = pd.DataFrame(raw_parsed)
df_profiles = df_raw.dropna(subset=['id']).drop_duplicates(subset=['id']).reset_index(drop=True)

df_skills = df_profiles.explode('skills').dropna(subset=['skills']).copy()
df_skills['skills_clean'] = df_skills['skills'].str.strip()

# Pengecekan tepat sasaran karena skills_clean sudah terstandardisasi
df_skills['is_soft'] = df_skills['skills_clean'].str.lower().isin(SOFT_SKILLS)

# ==============================================================================
# 6. RINGKASAN PREPROCESSING
# ==============================================================================
n_soft = int(df_skills['is_soft'].sum())
n_hard = int((~df_skills['is_soft']).sum())

print("=" * 55)
print("HASIL PREPROCESSING")
print("=" * 55)
print(f"- Data mentah masuk : {len(raw_documents)} dokumen")
print(f"- Total profil unik : {len(df_profiles)} alumni (duplikat dibersihkan)")
print(f"- Total baris skill : {len(df_skills)} baris")
print(f"- Total jenis skill : {df_skills['skills_clean'].nunique():,} skill unik")
print(f"  • Hard Skills     : {n_hard} baris ({round(n_hard/len(df_skills)*100, 1)}%)")
print(f"  • Soft Skills     : {n_soft} baris ({round(n_soft/len(df_skills)*100, 1)}%)")
print("=" * 55)

HASIL PREPROCESSING
- Data mentah masuk : 397 dokumen
- Total profil unik : 394 alumni (duplikat dibersihkan)
- Total baris skill : 5963 baris
- Total jenis skill : 2,658 skill unik
  • Hard Skills     : 5214 baris (87.4%)
  • Soft Skills     : 749 baris (12.6%)


## Dashboard Component 0: Key Performance Indicator

In [3]:
total_skill_unik = df_skills['skills_clean'].nunique()
rerata_skill_alumni = round(len(df_skills) / max(len(df_profiles), 1), 1)

alumni_terkini = df_profiles[df_profiles['latest_work_year'] >= 2026]
alumni_lama    = df_profiles[df_profiles['latest_work_year'] <= 2022]

keahlian_terkini = set(alumni_terkini.explode('skills')['skills'].dropna())
keahlian_lama    = set(alumni_lama.explode('skills')['skills'].dropna())

total_skill_tren = len(keahlian_terkini - keahlian_lama)

jumlah_skill_top = df_skills['skills_clean'].value_counts().iloc[0] if not df_skills.empty else 0
print("=" * 55)
print("KEY PERFORMANCE INDICATORS (KPI METRICS)")
print("=" * 55)
print(f"• Total Unique Skills        : {total_skill_unik:,} skill")
print(f"• Average Skills per Profile : {rerata_skill_alumni} keahlian / alumni")
print(f"• Trending Skills Count      : {total_skill_tren} keahlian era baru")
print(f"• Top Endorsed Skills        : {jumlah_skill_top} alumni")
print("=" * 55)

KEY PERFORMANCE INDICATORS (KPI METRICS)
• Total Unique Skills        : 2,658 skill
• Average Skills per Profile : 15.1 keahlian / alumni
• Trending Skills Count      : 1083 keahlian era baru
• Top Endorsed Skills        : 106 alumni


## Dashboard Component 1: Hard Skills vs Soft Skills (Bar Chart)

In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

selected_industry = ""  # Contoh lain: "Mining", "Oil and Gas", atau "" (Semua)
top_n = 10

df_s_filt = df_skills[df_skills['industry'].str.lower() == selected_industry.lower()] if selected_industry else df_skills

top_hard = df_s_filt[~df_s_filt['is_soft']]['skills'].value_counts().head(top_n).sort_values(ascending=True)
top_soft = df_s_filt[df_s_filt['is_soft']]['skills'].value_counts().head(top_n).sort_values(ascending=True)


fig_a = make_subplots(
    rows=1, cols=2,
    subplot_titles=(f"Top {top_n} Hard Skills", f"Top {top_n} Soft Skills"),
    shared_yaxes=False,
    horizontal_spacing=0.20
)

fig_a.add_trace(go.Bar(
    x=top_hard.values, y=top_hard.index,
    orientation='h', marker_color='#1d4ed8',
    name='Hard Skills', text=top_hard.values, textposition='auto'
), row=1, col=1)

fig_a.add_trace(go.Bar(
    x=top_soft.values, y=top_soft.index,
    orientation='h', marker_color='#10b981',
    name='Soft Skills', text=top_soft.values, textposition='auto'
), row=1, col=2)

fig_a.update_layout(
    title_text=f"<b>a. Pemetaan Hard Skills & Soft Skills Alumni</b> — Industri: {selected_industry or 'Semua'}",
    height=420,
    showlegend=False,
    margin=dict(l=20, r=20, t=60, b=20)
)
fig_a.show()

## Dashboard Component 2: Skills Distribution by Category

## Dashboard Component 3: Network of Top Skills

## Dashboard Component 4: Trending Skills

In [5]:
from collections import Counter, defaultdict
import pandas as pd
import plotly.graph_objects as go

# Parameter Konfigurasi Dashboard
top_n = 10
min_frequency = 10  # Ambang batas agar skill yang masuk benar-benar valid & berbobot
selected_industry = ''  # Contoh: "Mining", "Oil and Gas", atau "" (Semua)

# ==============================================================================
# 1. FILTER INDUSTRI BERDASARKAN DF_PROFILES
# ==============================================================================
profiles = df_profiles.copy()

if selected_industry:
    profiles = profiles[profiles['industry'].str.lower() == selected_industry.lower()]

# ==============================================================================
# 2. PEMBAGIAN BATCH TEMPORAL RELATIF (EARLIER 50% VS RECENT 50%)
# ==============================================================================
profiles_sorted = profiles.sort_values(by='fetched_at').reset_index(drop=True)

mid_point = len(profiles_sorted) // 2
earlier_profiles = profiles_sorted.iloc[:mid_point]
recent_profiles = profiles_sorted.iloc[mid_point:]

# ==============================================================================
# 3. PENGHITUNGAN FREKUENSI SKILL & LINKED ROLES
# ==============================================================================
skill_recent = Counter()
skill_earlier = Counter()
skill_linked_roles = defaultdict(Counter)

def aggregate_profile_data(df_subset, counter_target):
    for _, row in df_subset.iterrows():
        primary_role = (
            row['experiences'][0]['title'] 
            if row.get('experiences') and len(row['experiences']) > 0 
            else 'Specialist'
        )
        for s in row.get('skills', []):
            counter_target[s] += 1
            if primary_role:
                skill_linked_roles[s][primary_role] += 1

aggregate_profile_data(earlier_profiles, skill_earlier)
aggregate_profile_data(recent_profiles, skill_recent)

# ==============================================================================
# 4. KALKULASI & PENGURUTAN PERSENTASE TERBESAR KE TERKECIL
# ==============================================================================
all_skills = set(skill_earlier.keys()) | set(skill_recent.keys())
trending_list = []

for s_name in all_skills:
    c_rec = skill_recent[s_name]
    c_ear = skill_earlier[s_name]
    total_freq = c_rec + c_ear

    # Filter: frekuensi minimal 20 dan mengalami tren positif di batch recent
    if total_freq >= min_frequency and c_rec > c_ear:
        # Menghitung pangsa kemunculan pada window terbaru (0 - 100%)
        share_pct = (c_rec / total_freq) * 100

        top_roles = [r for r, _ in skill_linked_roles[s_name].most_common(2)]
        roles_text = ', '.join(top_roles) if top_roles else 'Professional Specialist'

        trending_list.append({
            'Skill Name': s_name,
            'Frequency': total_freq,
            'Share_Val': share_pct,
            'Recent Share (30d)': f'{share_pct:.1f}%',
            'Example linked roles': roles_text
        })

df_trending = pd.DataFrame(trending_list)

if not df_trending.empty:
    # Pengurutan langsung dari persentase terbesar ke terkecil
    df_trending = (
        df_trending.sort_values(by=['Share_Val', 'Frequency'], ascending=[False, False])
        .head(top_n)
        .reset_index(drop=True)
    )
else:
    df_trending = pd.DataFrame(columns=[
        'Skill Name', 'Frequency', 'Share_Val', 
        'Recent Share (30d)', 'Example linked roles'
    ])

# ==============================================================================
# 5. TABEL RINGKASAN INTERAKTIF PLOTLY
# ==============================================================================
industry_label = selected_industry if selected_industry else 'Semua'

cell_colors = (
    [['#f8fafc' if i % 2 == 0 else '#ffffff' for i in range(len(df_trending))]] * 4
    if len(df_trending) > 0
    else '#ffffff'
)

fig_d = go.Figure(
    data=[
        go.Table(
            columnwidth=[180, 90, 160, 310],
            header=dict(
                values=[
                    '<b>Skill Name</b>',
                    '<b>Total Frequency</b>',
                    '<b>Recent Share (30d)</b>',
                    '<b>Example linked roles</b>',
                ],
                fill_color='#0f172a',
                font=dict(color='white', size=12),
                align=['left', 'center', 'center', 'left'],
                height=36,
            ),
            cells=dict(
                values=[
                    df_trending['Skill Name'],
                    df_trending['Frequency'],
                    [f'<b>{val}</b>' for val in df_trending['Recent Share (30d)']],
                    df_trending['Example linked roles'],
                ],
                fill_color=cell_colors,
                font=dict(
                    color=['#0f172a', '#1e293b', '#16a34a', '#334155'], 
                    size=11
                ),
                align=['left', 'center', 'center', 'left'],
                height=34,
            ),
        )
    ]
)

fig_d.update_layout(
    title_text=(
        f'<b>d. Trending Skills: Pangsa Kemunculan 30 Hari Terakhir (0–100%)</b>'
        f' — Industri: {industry_label}'
    ),
    margin=dict(l=20, r=20, t=60, b=20),
    height=420,
)

fig_d.show()